# **01. Import libraries**

In [22]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, accuracy_score


# **02. Load dataset**

In [23]:
df = pd.read_csv('../data/Crop_recommendation.csv')

print("Dataset loaded successfully!")
print("Shape:", df.shape)
display(df.head())

Dataset loaded successfully!
Shape: (2200, 8)


,N,P,K,temperature,humidity,ph,rainfall,label
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,rice
2,60,55,44,23.004459,82.320763,7.840207,263.964248,rice
3,74,35,40,26.491096,80.158363,6.980401,242.864034,rice
4,78,42,42,20.130175,81.604873,7.628473,262.717340,rice


# **03. Feature Engineering**

In [24]:
# Technique 1: Feature Interaction 
df['temp_humidity_index'] = df['temperature'] * df['humidity']

# Technique 2: Aggregation 
df['total_npk'] = df['N'] + df['P'] + df['K']

# Technique 3: Binning 
df['ph_category'] = pd.cut(df['ph'], bins=[0, 5.5, 7.5, 14], labels=[0, 1, 2]).astype(int)

# Technique 4: Non-linear Transformation 
df['log_rainfall'] = np.log1p(df['rainfall'])

# Technique 5: Target Encoding 
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['label'])

print("Feature engineering completed successfully!")
display(df[['label', 'label_encoded']].drop_duplicates().head(5))

Feature engineering completed successfully!


,label,label_encoded
0,rice,20
100,maize,11
200,chickpea,3
300,kidneybeans,9
400,pigeonpeas,18


# **04. Prepare Features (X) and Target (y)**

In [25]:
X = df[['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall', 
        'temp_humidity_index', 'total_npk', 'ph_category', 'log_rainfall']]
y = df['label_encoded']

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (2200, 11)
y shape: (2200,)


# **05. Train/Test Split & Standardization**

In [26]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Technique 6: Feature Standardization 
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("Data standardized successfully!")

Training samples: 1760
Testing samples: 440
Data standardized successfully!


# **06. Model Training**

In [27]:
clf_crop = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clf_crop.fit(X_train_scaled, y_train)

print("Random Forest Classifier trained successfully!")

Random Forest Classifier trained successfully!


# **07. Model Evaluation**

In [28]:
y_pred = clf_crop.predict(X_test_scaled)

print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print("Classification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 99.32%

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        23
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        20
           3       1.00      1.00      1.00        26
           4       1.00      1.00      1.00        27
           5       1.00      1.00      1.00        17
           6       1.00      1.00      1.00        17
           7       1.00      1.00      1.00        14
           8       0.92      1.00      0.96        23
           9       1.00      1.00      1.00        20
          10       0.92      1.00      0.96        11
          11       1.00      1.00      1.00        21
          12       1.00      1.00      1.00        19
          13       1.00      0.96      0.98        24
          14       1.00      1.00      1.00        19
          15       1.00      1.00      1.00        17
          16       1.00      1.00      1

# **08. Export Models for FastAPI**

In [29]:
export_dir = '../models'
os.makedirs(export_dir, exist_ok=True)

joblib.dump(clf_crop, f'{export_dir}/crop_classifier.pkl')
joblib.dump(scaler, f'{export_dir}/crop_scaler.pkl')
joblib.dump(le, f'{export_dir}/crop_label_encoder.pkl')

print("Crop models exported successfully!")

Crop models exported successfully!
